In [3]:
pip install pandas mlxtend numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import mlxtend
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# ==========================================
# SIMULACIÓN DE DATOS DE LA LIBRERÍA
# ==========================================
# Filas = Transacciones de clientes / Columnas = Libros (1 = Comprado, 0 = No comprado)
data = {
    'El Quijote': [1, 0, 1, 1, 0, 1, 0, 1, 1, 0],
    'Cien anos de soledad': [1, 1, 1, 0, 1, 1, 1, 0, 1, 1],
    '1984': [0, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    'Rayuela': [1, 0, 0, 1, 0, 1, 0, 0, 1, 0],
    'El Principito': [0, 1, 1, 0, 1, 0, 1, 1, 0, 1],
    'Crimen y Castigo': [1, 1, 0, 1, 0, 1, 1, 0, 0, 1],
}

df_transacciones = pd.DataFrame(data)

# Dividir en conjunto de entrenamiento (para crear reglas) y prueba (para evaluar)
train_df = df_transacciones.iloc[:7]
test_df = df_transacciones.iloc[7:]


# ==========================================
# ACTIVIDAD 1: ANÁLISIS DE COMPRAS (Reglas de Asociación)
# ==========================================
print('--- 1. ANÁLISIS DE COMPRAS (REGLAS DE ASOCIACIÓN) ---')

# Encontrar itemsets frecuentes con un soporte mínimo del 30%
itemsets_frecuentes = apriori(train_df, min_support=0.3, use_colnames=True)

# Generar reglas de asociación evaluando la confianza (mínimo 50%)
reglas = association_rules(
    itemsets_frecuentes, metric='confidence', min_threshold=0.5
)

# Mostrar las reglas generadas ordenadas por lift
reglas_ordenadas = reglas.sort_values(by='lift', ascending=False)
print(
    reglas_ordenadas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False)
)


# ==========================================
# ACTIVIDAD 2: SISTEMA DE RECOMENDACIÓN
# ==========================================
print('\n--- 2. SISTEMA DE RECOMENDACIÓN (BASADO EN EL CARRITO) ---')


def recomendar_libros(carrito_actual, rules_df, top_n=2):
  recomendaciones = []

  for _, row in rules_df.iterrows():
    antecedentes = set(row['antecedents'])
    consecuentes = set(row['consequents'])

    # Si los antecedentes están contenidos en el carrito del usuario
    if antecedentes.issubset(set(carrito_actual)):
      for libro in consecuentes:
        # Evitar recomendar libros que ya están en el carrito
        if libro not in carrito_actual:
          recomendaciones.append((libro, row['lift']))

  # Filtrar duplicados conservando el mayor valor de Lift
  recomendaciones_unicas = {}
  for libro, lift in sorted(recomendaciones, key=lambda x: x[1], reverse=True):
    if libro not in recomendaciones_unicas:
      recomendaciones_unicas[libro] = lift

  return list(recomendaciones_unicas.keys())[:top_n]


# Prueba del sistema con un carrito de ejemplo
carrito_usuario = ['El Quijote']
sugerencias = recomendar_libros(carrito_usuario, reglas, top_n=2)

print(f'Libros actuales en el carrito: {carrito_usuario}')
print(f'Libros sugeridos por el sistema: {sugerencias}')


# ==========================================
# ACTIVIDAD 3: EVALUACIÓN DE EFECTIVIDAD
# ==========================================
print('\n--- 3. EVALUACIÓN DE EFECTIVIDAD ---')


def evaluar_sistema(df_test, rules_df, k=2):
  aciertos = 0
  total_evaluados = 0
  libros_recomendados_totales = set()

  for _, row in df_test.iterrows():
    # Obtener los libros que compró este usuario de prueba
    libros_comprados = list(df_test.columns[row == 1])

    if len(libros_comprados) < 2:
      continue  # Se necesitan al menos 2 libros para simular carrito y objetivo

    # Simulamos partición: mitad en carrito, mitad como libros esperados
    mitad = len(libros_comprados) // 2
    carrito_prueba = libros_comprados[:mitad]
    reales_esperados = set(libros_comprados[mitad:])

    # Obtener recomendaciones del sistema
    recomendaciones = recomendar_libros(carrito_prueba, rules_df, top_n=k)

    for rec in recomendaciones:
      libros_recomendados_totales.add(rec)

    # Verificar si al menos una recomendación coincide con lo que el usuario compró realmente (Hit)
    if any(rec in reales_esperados for rec in recomendaciones):
      aciertos += 1
    total_evaluados += 1

  # Métricas
  hit_rate = (
      (aciertos / total_evaluados) * 100 if total_evaluados > 0 else 0
  )
  catalogo_total = set(df_test.columns)
  cobertura = (
      (len(libros_recomendados_totales) / len(catalogo_total)) * 100
      if len(catalogo_total) > 0
      else 0
  )

  return hit_rate, cobertura


hit_rate, cobertura = evaluar_sistema(test_df, reglas, k=2)

print(f'Tasa de Acierto (Hit Rate / Precision@2): {hit_rate:.2f}%')
print(f'Cobertura del Catálogo: {cobertura:.2f}%')

--- 1. ANÁLISIS DE COMPRAS (REGLAS DE ASOCIACIÓN) ---
                          antecedents                           consequents  support  confidence     lift
                            (Rayuela)        (Crimen y Castigo, El Quijote) 0.428571    1.000000 2.333333
       (Crimen y Castigo, El Quijote)                             (Rayuela) 0.428571    1.000000 2.333333
         (1984, Cien anos de soledad)                       (El Principito) 0.571429    1.000000 1.750000
                      (El Principito)          (1984, Cien anos de soledad) 0.571429    1.000000 1.750000
                            (Rayuela)                          (El Quijote) 0.428571    1.000000 1.750000
                         (El Quijote)                             (Rayuela) 0.428571    0.750000 1.750000
                         (El Quijote)           (Rayuela, Crimen y Castigo) 0.428571    0.750000 1.750000
          (Rayuela, Crimen y Castigo)                          (El Quijote) 0.428571    1.000000 1

C:\Users\Pc\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
